In [1]:
import pickle
import numpy as np
import tensorflow as tf
import keras
from cnn_base import build_t1_quantized, build_qkeras_model
import os


TRAIN_PICKLE_PATH = "./dataset/sat6_train.pkl"
TEST_PICKLE_PATH = "./dataset/sat6_test.pkl"
DATA_DIR = "./dataset/sat-6-full.mat"
BATCH_SIZE = 512
EPOCHS = 30
LEARNING_RATE = 3e-4
 
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

def trace_layers(config):
    config['Model']['TraceOutput'] = True
    for layer in config['LayerName']:
        config['LayerName'][layer]['Trace'] = True


def print_dict(d, indent=0):
    for key, value in d.items():
        print('  ' * indent + str(key), end='')
        if isinstance(value, dict):
            print()
            print_dict(value, indent + 1)
        else:
            print(':' + ' ' * (20 - len(key) - 2 * indent) + str(value))
 
def load_pickle_data(path):
    print(f"Lendo {path} ...")
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data["X"], data["y"]

2026-09-14 20:16:25.046590: I tensorflow/core/util/port.cc:110] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-14 20:16:25.047511: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-14 20:16:25.064660: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-14 20:16:25.065296: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-09-14 20:16:25.388108: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:3

In [2]:
X_train_all, y_train_all = load_pickle_data(TRAIN_PICKLE_PATH)
X_test, y_test = load_pickle_data(TEST_PICKLE_PATH)

X_train_all = X_train_all.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

val_idx = int(0.75 * len(X_train_all))
X_train, y_train = X_train_all[:val_idx], y_train_all[:val_idx]
X_val, y_val = X_train_all[val_idx:], y_train_all[val_idx:]

n_classes = 6
input_shape = X_train.shape[1:]
train_size = len(X_train)
input_shape, train_size

Lendo ./dataset/sat6_train.pkl ...
Lendo ./dataset/sat6_test.pkl ...


((32, 32, 4), 243000)

# Train T1 base model

In [ ]:
model = build_t1_quantized()
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.CategoricalAccuracy(name="accuracy")
    ],
)
callbacks = [
    keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1),
]

model.fit(X_train, y_train, epochs=30, validation_data=(X_val, y_val), callbacks=callbacks)


os.makedirs('models', exist_ok=True)
os.makedirs('models/base_model_t1', exist_ok=True)
base_path = './models/base_model_t1/'
model.save(base_path + '/t1_model.keras')
model.save(base_path + '/t1_model.h5')
model.save_weights(base_path + '/t1_weights.h5')

with open(base_path + '/t1_model.json', 'w') as f:
    f.write(model.to_json())

# Another Models 

In [ ]:
filters_per_conv_layer = [20, 8]
#neurons_per_dense_layer = [42]

model = build_qkeras_model(input_shape=input_shape, n_classes=n_classes, filters_per_conv_layer=filters_per_conv_layer)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=[
        tf.keras.metrics.CategoricalAccuracy(name="accuracy")
    ],
)
callbacks = [
    keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1),
]

model.fit(X_train, y_train, epochs=30, validation_data=(X_val, y_val), callbacks=callbacks)


os.makedirs('models', exist_ok=True)
model_name = 'qkeras_model'
os.makedirs(f'models/{model_name}', exist_ok=True)
base_path = f'./models/{model_name}/'
model.save(base_path + '/model.keras')
model.save(base_path + '/model.h5')
model.save_weights(base_path + '/weights.h5')

with open(base_path + '/model.json', 'w') as f:
    f.write(model.to_json())

Epoch 1/30
7594/7594 [==============================] - 23s 3ms/step - loss: 0.1353 - accuracy: 0.9501 - val_loss: 0.0744 - val_accuracy: 0.9731 - lr: 3.0000e-04
Epoch 2/30
7594/7594 [==============================] - 22s 3ms/step - loss: 0.0755 - accuracy: 0.9733 - val_loss: 0.0546 - val_accuracy: 0.9827 - lr: 3.0000e-04
Epoch 3/30
7594/7594 [==============================] - 22s 3ms/step - loss: 0.0621 - accuracy: 0.9784 - val_loss: 0.0604 - val_accuracy: 0.9799 - lr: 3.0000e-04
Epoch 4/30
7594/7594 [==============================] - 22s 3ms/step - loss: 0.0537 - accuracy: 0.9810 - val_loss: 0.0471 - val_accuracy: 0.9838 - lr: 3.0000e-04
Epoch 5/30
7594/7594 [==============================] - 22s 3ms/step - loss: 0.0481 - accuracy: 0.9833 - val_loss: 0.0453 - val_accuracy: 0.9849 - lr: 3.0000e-04
Epoch 6/30
7594/7594 [==============================] - 22s 3ms/step - loss: 0.0430 - accuracy: 0.9849 - val_loss: 0.0431 - val_accuracy: 0.9859 - lr: 3.0000e-04
Epoch 7/30
7594/7594 [======

/home/fauri/Documentos/hls4ml/.venv_hls4ml_t/lib/python3.10/site-packages/keras/src/engine/training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


: 